In [118]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [119]:
# import

df = pd.read_csv('./datasets/titanic.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [120]:
cols = [
    'Cabin',
    'Embarked'
]

df[cols] = df[cols].fillna('missing')
df['Age'] = df['Age'].fillna(df['Age'].mean())

df['Cabin_prefix'] =  df['Cabin'].str[0]
df['name_word_count'] = df['Name'].str.split().str.len()
df['name_len'] = df['Name'].str.len()
df['name_surname'] = df['Name'].str.split(',').str[0]

df['Ticket_prefix'] = df['Ticket'].str.split().str[0]
df['family_count'] = df['SibSp'] + df['Parch']
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Cabin_prefix,name_word_count,name_len,name_surname,Ticket_prefix,family_count
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,missing,S,m,4,23,Braund,A/5,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,C,7,51,Cumings,PC,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,missing,S,m,3,22,Heikkinen,STON/O2.,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,C,7,44,Futrelle,113803,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,missing,S,m,4,24,Allen,373450,0


In [121]:
df['name_surname'].value_counts()

name_surname
Andersson          9
Sage               7
Skoog              6
Panula             6
Carter             6
                  ..
Vander Cruyssen    1
Gale               1
van Melkebeke      1
Bonnell            1
Adams              1
Name: count, Length: 667, dtype: int64

In [122]:
drop_cols = ['Cabin', 'Ticket', 'Name', 'PassengerId']

df = df.drop(columns=drop_cols)

In [123]:
cat_cols = [
    'Pclass',
    'Sex',
    'Embarked',
    'Cabin_prefix',
    'SibSp',
    'Parch',
    'Ticket_prefix',
    'name_surname'
]

df[cat_cols] = df[cat_cols].astype('category')

In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Survived         891 non-null    int64   
 1   Pclass           891 non-null    category
 2   Sex              891 non-null    category
 3   Age              891 non-null    float64 
 4   SibSp            891 non-null    category
 5   Parch            891 non-null    category
 6   Fare             891 non-null    float64 
 7   Embarked         891 non-null    category
 8   Cabin_prefix     891 non-null    category
 9   name_word_count  891 non-null    int64   
 10  name_len         891 non-null    int64   
 11  name_surname     891 non-null    category
 12  Ticket_prefix    891 non-null    category
 13  family_count     891 non-null    int64   
dtypes: category(8), float64(2), int64(4)
memory usage: 94.0 KB


In [125]:
X = df.drop(columns='Survived')
y = df['Survived']

In [128]:
from tabnanny import verbose

from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create and train LightGBM model
model = lgb.LGBMClassifier(
    random_state=42,
    verbose=-1,
    is_unbalance=True,
    #scale_pos_weight=3,  # ← Key for precision
    n_estimators=200,
    learning_rate=0.05,
    max_depth=7,
)

model.fit(X_train,
          y_train,
          categorical_feature=cat_cols
          )

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.8324022346368715
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       105
           1       0.80      0.80      0.80        74

    accuracy                           0.83       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.83      0.83      0.83       179

